# Đề tài 2: Phân Đoạn Ảnh Y Tế (Kvasir-SEG) — Thiết Lập Trên Kaggle


In [ ]:
# 1. Kiểm tra môi trường Kaggle và phần cứng GPU (T4 x2 / P100)
!nvidia-smi

import os
import torch

print("\n--- Thông tin Môi trường & GPU ---")
print(f"Kaggle Runtime   : {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Local / Other')}")
print(f"Working Dir      : {os.getcwd()}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"GPU Count        : {num_gpus}")
    for idx in range(num_gpus):
        vram_gb = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
        print(f"  [GPU {idx}] {torch.cuda.get_device_name(idx)} ({vram_gb:.2f} GB VRAM)")
else:
    print("Error")


## 2. Clone / Đồng Bộ Repository Nhóm Vào `/kaggle/working`
Tải mã nguồn dự án từ [GitHub DL-Group7](https://github.com/daoviet2/DL-Group7.git) vào thư mục ghi được `/kaggle/working/DL-Group7`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/daoviet2/DL-Group7.git"
WORK_BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_BASE, "DL-Group7")

if os.path.exists("configs/base.yaml"):
    print("[Git] Đang ở sẵn trong thư mục gốc dự án:", os.getcwd())
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print("[Git] Đã chuyển vào thư mục dự án:", os.getcwd())
    subprocess.run(["git", "pull"], check=True)
else:
    os.chdir(WORK_BASE)
    print(f"[Git] Đang clone repository từ {REPO_URL} vào {WORK_BASE} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    print("[Git] Thư mục làm việc hiện tại:", os.getcwd())

subprocess.run(["git", "status", "-s"], check=True)


## 3. Cài Đặt Thư Viện Phụ Thuộc Cho Cả 3 Nhánh (CNN, SegFormer, SAM)
Trên Kaggle đã cài sẵn PyTorch + CUDA, NumPy, Pandas và OpenCV; lệnh dưới chỉ cài bổ sung các gói còn thiếu trong `requirements.txt` (`segmentation-models-pytorch`, `evaluate`, v.v.).


In [ ]:
!pip install -r requirements.txt -q

import cv2
import albumentations as A
import segmentation_models_pytorch as smp
import transformers
import pandas as pd
import numpy as np

print("--- Xác nhận phiên bản thư viện trên Kaggle ---")
print("Albumentations :", A.__version__)
print("OpenCV         :", cv2.__version__)
print("SMP (U-Net)    :", smp.__version__)
print("Transformers   :", transformers.__version__)
print("[OK] Môi trường thư viện đã sẵn sàng!")


## 4. Kết Nối Dữ Liệu Kvasir-SEG (`/kaggle/input` hoặc Tự Động Tải)
Hệ thống tự động quét `/kaggle/input` nếu bạn đã gắn dataset trên Kaggle; nếu chưa có sẽ tự động tải và giải nén vào `./Kvasir-SEG`.


In [ ]:
import os
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from scripts.download_data import download_and_extract
from src.datasets.kvasir import resolve_kvasir_root

data_dir = download_and_extract()
resolved_dir = resolve_kvasir_root(data_dir)

img_count = len([f for f in os.listdir(os.path.join(resolved_dir, "images")) if f.lower().endswith((".jpg", ".png"))])
mask_count = len([f for f in os.listdir(os.path.join(resolved_dir, "masks")) if f.lower().endswith((".jpg", ".png"))])

print(f"\n[Dữ liệu] Đường dẫn đang dùng : {resolved_dir}")
print(f"[Dữ liệu] Images: {img_count}/1000 | Masks: {mask_count}/1000")
assert img_count == 1000 and mask_count == 1000, "Dữ liệu chưa đủ 1000 cặp ảnh-mask!"
print("[OK] Bộ dữ liệu Kvasir-SEG toàn vẹn 100%.")


## 5. Xác Nhận Bộ Chia Tập Phân Tầng Cố Định (Stratified 80/10/10)
Mọi mô hình của cả 3 thành viên bắt buộc dùng chung `splits/train.txt` (800), `splits/val.txt` (100) và `splits/test.txt` (100).


In [ ]:
if not os.path.exists("splits/metadata.csv"):
    !python scripts/prepare_splits.py
else:
    df_meta = pd.read_csv("splits/metadata.csv")
    print(f"[Splits] Đã nạp splits/metadata.csv ({len(df_meta)} mẫu):")
    print(df_meta.groupby(["split", "size_category"]).size().unstack(fill_value=0))


## 6. Chạy Kiểm Thử Tự Động Pipeline Dữ Liệu (Sanity Check)
Xác thực `KvasirSegDataset`, kích thước tensor `[B, 3, 256, 256]`, tính nhị phân `{0.0, 1.0}` của mask và tọa độ `bbox` cho SAM.


In [ ]:
!python scripts/sanity_check_data.py


## 7. Trực Quan Hóa Mini-Batch Trên Kaggle Notebook
Hiển thị trực tiếp 4 mẫu từ `train_loader` gồm: **Ảnh gốc kèm Ground Truth BBox | Ground Truth Mask | Overlay vùng Polyp**.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from src.datasets.kvasir import get_dataloaders
from src.utils.config import load_config
from src.utils.vis import create_overlay, draw_bounding_box
from src.datasets.transforms import denormalize_image

cfg = load_config("configs/base.yaml")
cfg_demo = cfg.copy()
cfg_demo["data"] = cfg["data"].copy()
cfg_demo["data"]["num_workers"] = 2
cfg_demo["data"]["batch_size"] = 4

train_loader, val_loader, test_loader = get_dataloaders(cfg_demo)
batch = next(iter(train_loader))

images = batch["image"]
masks = batch["mask"]
sample_ids = batch["id"]
bboxes = batch["bbox"]

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    img_denorm = denormalize_image(images[i])
    mask_np = masks[i, 0].numpy()

    img_box = draw_bounding_box(img_denorm, bboxes[i], color=(255, 50, 50), thickness=2)
    overlay = create_overlay(img_denorm, mask_np, color=(0, 230, 80), alpha=0.45)

    axes[i, 0].imshow(img_box)
    axes[i, 0].set_title(f"Ảnh gốc (BBox): {sample_ids[i]}", fontsize=11)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask", fontsize=11)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay Phân Đoạn", fontsize=11)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()


## 8. Kiểm Thử Forward Pass Trên GPU Kaggle & Thư Mục Xuất Kết Quả
Mọi file log `.csv`, hình ảnh và checkpoint `.pth` lưu trong `/kaggle/working/DL-Group7/outputs` sẽ xuất hiện tại mục **Output** bên phải của Kaggle để tải về hoặc lưu qua *Save Version*.


In [ ]:
if torch.cuda.is_available():
    import time
    device = torch.device("cuda")

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(device)

    real_images = batch["image"].to(device)
    torch.cuda.synchronize()
    start_time = time.time()
    with torch.no_grad():
        logits = model(real_images)
    torch.cuda.synchronize()
    elapsed = (time.time() - start_time) * 1000

    allocated_mem = torch.cuda.memory_allocated(device) / (1024 ** 2)
    print("--- Kiểm Thử Forward Pass GPU Kaggle Hoàn Tất ---")
    print(f"Batch đầu vào        : {real_images.shape}")
    print(f"Logits đầu ra        : {logits.shape}")
    print(f"Thời gian Forward    : {elapsed:.2f} ms")
    print(f"VRAM cấp phát        : {allocated_mem:.2f} MB")
    print(f"Thư mục lưu Outputs  : {os.path.abspath('outputs')}")
    print("[OK] Môi trường Kaggle GPU đã sẵn sàng 100% cho cả 3 thành viên!")
else:
    print("Đang chạy trên CPU. Hãy bật GPU T4 x2 / P100 trong Kaggle Session options khi huấn luyện!")
